# 교안 01-6: 종합 실습 - 데이터 분석 자동화 파이프라인

## 핵심 목표

조회는 DB, 계산과 그래프는 코드, 저장은 파일 서버.
세 갈래를 한 에이전트에 붙여 **질문 한 문장에서 리포트 파일까지** 자동으로 잇는다.

## 학습 순서

1. 세 서버(DB·코드 실행·파일시스템)의 도구를 **한 목록으로** 받기(이름 충돌은 접두사로)
2. 실습 1: 그래프를 그려 PNG 로 저장하게 하기
3. 실습 2: 리포트를 마크다운 파일로 저장하게 하기

## 앞 실습과의 관계

01(파일시스템)·04(SQLite)·05(코드 실행)에서 **하나씩** 붙여 본 서버를 여기서 **한꺼번에** 붙입니다.

새로 배우는 것은 둘입니다.

- **여러 서버를 한 클라이언트에** 등록하고 이름 충돌을 접두사로 막는 방법
- 같은 도구 묶음이라도 **프롬프트와 도구 목록을 어떻게 주느냐에 따라 결과가 달라진다**는 것.
  실습 1과 2는 붙이는 도구도 시키는 일도 다릅니다. 그 차이를 나란히 놓고 봅니다.

## 쓰는 MCP 서버와 공식 문서

| 서버 | 실행 | 맡는 일 | 공식 문서 |
|---|---|---|---|
| SQLite `mcp-server-sqlite` | `uvx` | 집계·조인 | https://pypi.org/project/mcp-server-sqlite/ |
| 코드 실행 `mcp-server-code-runner` | `npx` | 증감률·이동평균 | https://github.com/formulahendry/mcp-server-code-runner |
| 파일시스템 `@modelcontextprotocol/server-filesystem` | `npx` | 리포트 저장 | https://github.com/modelcontextprotocol/servers/tree/main/src/filesystem |

## 준비물

- **Node.js**(`npx`)와 **uv**(`uvx`), **`OPENAI_API_KEY`**(일차 폴더의 `.env`)
- 서버를 셋 띄우므로 첫 실행은 특히 오래 걸립니다.

---
## 준비

앞 실습에서 하나씩 만든 것을 그대로 가져옵니다. 새로 만드는 것은 없습니다.

In [ ]:
import sys
from pathlib import Path

# 노트북에는 __file__ 이 없다. 주피터는 노트북이 있는 폴더를 작업 폴더로 잡아 주므로 그 위가 일차 폴더다.
DAY_DIR = Path.cwd().parent        # 일차 폴더(day21). 아래 경로들의 기준점
sys.path.append(str(DAY_DIR))   # 일차 폴더의 utils.py 를 쓴다

import platform

from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_openai import ChatOpenAI

from utils import (child_env, chinook_db_path, load_api_key,
                   print_trajectory, quiet_stdio_logs)

quiet_stdio_logs()          # 코드 실행 서버가 stdout 에 섞어 보내는 안내문 때문에 나는 긴 경고를 끈다
load_api_key(DAY_DIR)       # 모델을 부르므로 키를 맨 앞에서 확인한다

DATA_DIR = DAY_DIR / "data"
DB_PATH = chinook_db_path(DATA_DIR)   # 04 번에서 쓴 그 DB(data 폴더에 있다)
CHILD_ENV = child_env()                 # 05 번에서 쓴 그 환경 변수

# 산출물 경로는 우리가 만들어 넘긴다. 절대경로라야 코드 실행 서버가 어느 폴더에서 돌든 같은 자리에 남는다.
CHART_PATH = DAY_DIR / "output" / "연도별_매출.png"
REPORT_PATH = DAY_DIR / "output" / "매출_리포트.md"

# 한글 폰트 이름은 OS 마다 다르다. 코드 실행 서버는 같은 컴퓨터에서 도니까 여기서 정해 넘긴다.
FONT = {"Windows": "Malgun Gothic", "Darwin": "AppleGothic"}.get(platform.system(), "NanumGothic")

# 두 실습이 함께 쓰는 앞부분. 어떤 일을 어느 도구에 맡길지를 못 박는다.
BASE_PROMPT = (
    "너는 데이터 분석 도우미다. 표의 집계는 db_read_query 로 SQL 을 실행해 구하고, "
    "그 결과를 가공하는 계산은 code_run-code 로 한다. 숫자를 암산하거나 지어내지 않는다. "
    "코드의 결과에 대한 마지막 줄은 반드시 print 로 출력한다. "
    "결과가 비어 있으면 print 를 빠뜨린 것이니 print 를 넣어 다시 실행한다. "
    "표나 열 이름이 확실하지 않으면 db_list_tables 와 db_describe_table 로 먼저 확인한다. "
)

---
## 서버 설정 세 개

세 설정 모두 앞 실습에서 한 줄씩 뜯어본 것입니다. 여기서는 **역할 분담**만 다시 확인합니다.

| 서버 | 맡는 일 | 왜 이 서버인가 |
|---|---|---|
| `db` | 집계·필터·조인 | 412행을 다 끌고 오지 않고 **5행으로 줄여서** 받는다 |
| `code` | 증감률·이동평균 | SQL 로 쓰기 번거로운 가공은 파이썬이 읽기도 고치기도 쉽다 |
| `files` | 리포트 저장 | 열어 준 폴더 밖으로 못 나가므로 **경계가 이미 그어져 있다** |

In [ ]:
# 04 번에서 쓴 그 설정이다. DB 파일 하나만 열어 준다.
SQLITE = {
    "command": "uvx",
    "args": ["--with", "mcp==1.9.4",         # mcp 버전 고정. 최신으로 띄우면 서버가 바로 죽는다
             "--from", "mcp-server-sqlite",
             "mcp-server-sqlite",
             "--db-path", str(DB_PATH)],     # 열어 줄 DB 파일 하나
    "transport": "stdio",
}

# 05 번에서 쓴 그 설정이다. env 로 코드를 실행할 환경을 정해 준다.
CODE_RUNNER = {
    "command": "npx",
    "args": ["-y", "mcp-server-code-runner"],
    "transport": "stdio",
    "env": CHILD_ENV,                        # 서버가 python 을 찾게 하는 환경 변수
}

# 01 번에서 쓴 그 설정이다. 리포트를 저장할 폴더를 열어 준다.
FILESYSTEM = {
    "command": "npx",
    "args": ["-y", "@modelcontextprotocol/server-filesystem",
             str(DAY_DIR)],                  # 이 폴더 밖은 건드리지 못한다
    "transport": "stdio",
}

---
## 1. 세 서버의 도구를 한 목록으로 받기

지금까지는 서버를 **하나씩** 붙였습니다. 딕셔너리에 나란히 적으면 **한 클라이언트가 셋을 다 띄웁니다**.

문제가 하나 생깁니다. 서버가 여럿이면 **같은 이름의 도구가 겹칠 수 있습니다**(파일 서버의 `read_file` 과
다른 서버의 `read_file` 처럼). 그래서 **`tool_name_prefix=True`** 를 줍니다.
우리가 붙인 별명(`db`·`code`·`files`)이 도구 이름 앞에 붙습니다.

| 별명 | 붙은 이름 예 | 맡는 일 |
|---|---|---|
| `db` | `db_read_query`·`db_list_tables` | 집계·조회 |
| `code` | `code_run-code` | 계산·그래프 |
| `files` | `files_write_file`·`files_read_text_file` | 저장·읽기 |

접두사는 충돌을 막는 것 말고 쓸모가 하나 더 있습니다. **이름만 보고 서버 단위로 도구를 고를 수 있습니다.**
뒤에서 그 방식으로 권한을 좁힙니다.

In [ ]:
print("서버 세 개를 띄우는 중입니다(첫 실행은 오래 걸립니다)...")
client = MultiServerMCPClient(
    {"db": SQLITE, "code": CODE_RUNNER, "files": FILESYSTEM},
    tool_name_prefix=True,      # db_·code_·files_ 접두사를 붙인다. 서버끼리 도구 이름이 겹쳐도 충돌하지 않는다
)
tools = await client.get_tools()
by_name = {tool_item.name: tool_item for tool_item in tools}

print(f"도구 {len(tools)}개")
print(" ", ", ".join(sorted(by_name)))

---
## 2. 실습 1: 그래프를 그려 저장하게 하기

그림은 **코드 실행 서버**가 그립니다. 파일 저장 도구는 넣지 않습니다.
권한은 말이 아니라 **목록으로** 줍니다. 넘기지 않은 도구는 모델이 존재조차 모릅니다.

그림에만 붙는 조건이 셋이라 프롬프트에 미리 적어 둡니다.

| 무엇 | 왜 |
|---|---|
| `matplotlib.use("Agg")` | 저쪽은 **창을 띄울 수 없는 프로세스**다. 화면 대신 파일로만 그린다 |
| 한글 폰트 지정 | 기본 폰트에 한글이 없어 제목이 네모(□□□)로 나온다. 이름은 OS 마다 다르다 |
| 경고 끄기 | 이 서버는 **경고가 하나라도 나면 표준 출력을 통째로 버린다**. `print` 한 값이 사라져 모델이 실패로 오해하고 같은 코드를 계속 다시 보낸다 |

In [ ]:
# timeout 을 준다. 기본값은 10분이라 응답이 늦으면 멈춘 것과 구별되지 않는다.
model = ChatOpenAI(model="gpt-4o-mini", temperature=0, timeout=60)

chart_tools = [t for t in tools if t.name in
               {"db_read_query", "db_list_tables", "db_describe_table", "code_run-code"}]
print("에이전트에 붙인 도구:", [t.name for t in chart_tools])

chart_agent = create_agent(
    model,
    chart_tools,
    system_prompt=BASE_PROMPT + (
        "그래프는 matplotlib 으로 그리되 matplotlib.use('Agg') 로 창을 띄우지 않는다. "
        f"한글이 깨지지 않게 plt.rcParams['font.family'] 를 '{FONT}' 로 지정한다. "
        "코드 맨 위에서 warnings.filterwarnings('ignore') 로 경고를 끈다. "
        "경고가 하나라도 나면 이 서버는 표준 출력을 버려서 print 한 값이 사라진다. "
        "code_run-code 는 한 번에 하나씩만 부른다. 이 서버는 모든 코드를 같은 임시 파일에 쓰므로 "
        "동시에 두 번 부르면 서로의 코드를 덮어써 실패한다. "
        "저장을 마치면 os.path.getsize 로 파일 크기를 재서 경로와 함께 print 한다. 0 이면 저장에 실패한 것이다."
    ),
    # 반복에 빠져도 상한에서 스스로 끝난다(exit_behavior="end" 라 예외 없이 기록이 돌아온다).
    middleware=[ModelCallLimitMiddleware(run_limit=15, exit_behavior="end")],
)

chart_question = (
    "invoices 표에서 연도별 매출 합계를 구하고, 연도별 막대그래프를 그려서 "
    f"'{CHART_PATH}' 에 저장해 줘. 경로는 이 문자열을 그대로 써 줘."
)
print("질문:", chart_question, "\n")

print_trajectory(await chart_agent.ainvoke({"messages": chart_question}))

In [ ]:
# 모델의 '저장했습니다' 라는 말이 아니라 파일로 확인한다.
print("그림이 생겼나?:", CHART_PATH.exists())

---
## 3. 실습 2: 리포트를 파일로 저장하게 하기

같은 DB·코드 도구에 **파일 쓰기 도구 하나**를 더합니다.
시키는 일이 달라지면 붙일 도구도 달라진다는 것을 실습 1과 나란히 놓고 보세요.

질문 한 문장 안에 **조회·계산·서식·저장**이 모두 들어 있습니다.
어떤 도구를 어떤 순서로 고르는지 기록으로 확인하세요.

In [ ]:
report_tools = chart_tools + [t for t in tools if t.name == "files_write_file"]
print("에이전트에 붙인 도구:", [t.name for t in report_tools])

report_agent = create_agent(
    model,
    report_tools,
    system_prompt=BASE_PROMPT + (
        "파일로 저장할 때는 files_write_file 을 쓴다. "
        "표는 마크다운 표로 정리하고, 수치는 조회·계산한 값만 쓴다."
    ),
    middleware=[ModelCallLimitMiddleware(run_limit=15, exit_behavior="end")],
)

report_question = (
    "invoices 표에서 연도별 매출 합계를 구하고, 전년 대비 증감률까지 계산해 줘. "
    f"결과를 표로 정리한 마크다운 리포트를 '{REPORT_PATH}' 에 저장해 줘. "
    "경로는 이 문자열을 그대로 써 줘."
)
print("질문:", report_question, "\n")

print_trajectory(await report_agent.ainvoke({"messages": report_question}))

In [ ]:
print("리포트가 생겼나?:", REPORT_PATH.exists())
if REPORT_PATH.exists():
    print("-" * 40)
    print(REPORT_PATH.read_text(encoding="utf-8")[:400])

> 기록을 읽는 요령입니다. `db_read_query` → `code_run-code` → `files_write_file` 순서로 갔다면
> 조회 → 계산 → 저장을 모델이 스스로 이어 붙인 것입니다.
> 순서가 달라져도 결과가 맞을 수 있습니다. 볼 것은 **각 숫자가 도구에서 나온 값인가** 입니다.

### 🖐️ 함께 따라하기: 도구를 하나 빼면 무엇이 달라지나

권한을 목록으로 준다는 말을 **실험으로** 확인합니다.

1. `report_tools` 에서 **`files_write_file` 을 뺀** 도구 목록으로 새 에이전트를 만드세요(시스템 프롬프트는 같게).
2. 위와 **같은 질문**(저장까지 요구하는 문장)을 던지세요.
3. `print_trajectory()` 로 기록을 찍고, 모델이 저장을 어떻게 처리했는지 보세요.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) report_tools 에서 files_write_file 을 뺀 목록으로 새 에이전트를 만든다
# 2) 저장까지 요구하는 같은 질문을 던진다
# 3) print_trajectory 로 기록을 찍어 파일 쓰기 호출이 있는지 확인한다